In [1]:
%%writefile resnet.py
import torch
import torch.nn as nn
from torchvision.models import resnet50, ResNet50_Weights

class ResNet(nn.Module):
  def __init__(self):
    super().__init__()
    self.rete = resnet50(weights = ResNet50_Weights.DEFAULT)      #crea la rete
    self.features = {}
    for par in self.rete.parameters():
      par.requires_grad = False           #blocca l'aggiornamneto dei pesi
    self.rete.layer2.register_forward_hook(self.getFeatures('layer2'))
    self.rete.layer3.register_forward_hook(self.getFeatures('layer3'))
    self.rete.eval();                         #elimina il drop out e altre tecniche di overfitting

  def getFeatures(self, layerName):
    def extractor(mod, input, output):
      self.features[layerName] = output  #per avere informazioni sul layer
    return extractor

  def forward(self, x):
    _ = self.rete(x)          #scartatiamo l'out finale
    return self.features     #restituisce le features esatte

Overwriting resnet.py


In [2]:

%%writefile patchcore.py
import torch
import torch.nn.functional as F
from sklearn.random_projection import SparseRandomProjection

class PatchCore:
  def __init__(self, fe, d):
    self.extractor = fe
    self.device = d
    self.memory_bank = None

  def fit(self, dl):
    fea_l2 = []                      #crea le liste vuote per le features
    fea_l3 = []
    self.extractor.eval();           #evita di aggiornare i pesi
    with torch.no_grad():
      for images in dl:
        images = images.to(self.device)
        out = self.extractor(images)
        fea_l2.append(out['layer2'].cpu())
        fea_l3.append(out['layer3'].cpu())

      fea_l2 = torch.cat(fea_l2, dim=0)
      fea_l3 = torch.cat(fea_l3, dim=0)       #creiamo due grandi tensori lungo la prima dim
      fea_l3NEW = F.interpolate(fea_l3, size=(fea_l2.shape[2], fea_l2.shape[3]), mode='bilinear', align_corners=False)    #il lay3 è piccolo, va ridimensionacome come il 2
      pc_fea = torch.cat([fea_l2, fea_l3NEW], dim=1) #incolliamo i tensori lungo la dimensione die canali
      B, C, H, W = pc_fea.shape
      self.memory_bank = pc_fea.permute(0, 2, 3, 1).reshape(-1, C)
    print(f"{self.memory_bank.shape[0]} vettori (patch).")

  def subsampling(self, perc):
    mapper = SparseRandomProjection(n_components=128, random_state=42)        #il mapper schiaccia i tenssori mantendno le distanze invariate, 128 canali
    bank_np = self.memory_bank.numpy()                                          #converto il mapper in numpy
    bank_sparse = mapper.fit_transform(self.memory_bank.numpy())     #sparsifica i tensori
    bank_sparse = torch.tensor(bank_sparse)
    num_target = int(self.memory_bank.shape[0] * perc)
    indices = torch.randperm(self.memory_bank.shape[0])[:num_target]        #estraggo gli indici dei vettori in mada da avere un dataset significatico
    self.memory_bank = self.memory_bank[indices]
    print(f"{self.memory_bank.shape[0]} nuova dimensione")

  def predict(self, image_tensor):
    self.extractor.eval()
    with torch.no_grad():
      image_tensor = image_tensor.to(self.device)
      out = self.extractor(image_tensor)        #estraggo patch
      f2 = out['layer2'].cpu()
      f3 = out['layer3'].cpu()
      f3NEW = F.interpolate(f3, size=(f2.shape[2], f2.shape[3]), mode='bilinear', align_corners=False)    #il lay3 è piccolo, va ridimensionacome come il 2
      pc_fea = torch.cat([f2, f3NEW], dim=1)    #creiamo due grandi tensori lungo la prima dim
      B, C, H, W = pc_fea.shape
      test_patches = pc_fea.permute(0, 2, 3, 1).reshape(-1, C)     #incolliamo i tensori lungo la dimensione die canali
      distanze = torch.cdist(test_patches, self.memory_bank)     #calolo le dist
      min_distanze, _ = torch.min(distanze, dim=1)            #trovo la distanza min (le features più simili)
      score_anomalia = torch.max(min_distanze).item()         #trovo la dist max (stranezza peggiore)
      return score_anomalia

  def get_heatmap(self, image_tensor):
    self.extractor.eval()
    with torch.no_grad():
      image_tensor = image_tensor.to(self.device)
      out = self.extractor(image_tensor)
      f2 = out['layer2'].cpu()
      f3 = out['layer3'].cpu()
      f3NEW = F.interpolate(f3, size=(f2.shape[2], f2.shape[3]), mode='bilinear', align_corners=False)
      pc_fea = torch.cat([f2, f3NEW], dim=1)
      B, C, H, W = pc_fea.shape

      test_patches = pc_fea.permute(0, 2, 3, 1).reshape(-1, C)
      distanze = torch.cdist(test_patches, self.memory_bank)
      min_distanze, _ = torch.min(distanze, dim=1)
      score_anomalia = torch.max(min_distanze).item()


      heatmap = min_distanze.reshape(H, W).numpy()

      return score_anomalia, heatmap

Overwriting patchcore.py


In [3]:
%%writefile dataset.py
import os
from PIL import Image
from torch.utils.data import Dataset
import torchvision.transforms as transforms

pp = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),        #da la forma all'immaigne per darrla alla rete
    transforms.ToTensor(),           #trasforma in un tensore
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

class immagini:
  def __init__(self, metal_nut, trasnform = pp):
    self.transform = trasnform
    self.image_paths = []
    for root, dirs, files in os.walk(metal_nut):
      for file in files:
        if file.lower().endswith(('.png', '.jpg', '.jpeg')):
          self.image_paths.append(os.path.join(root, file))
    if len(self.image_paths) == 0:
      print("0")

  def __len__(self):
    return len(self.image_paths)

  def __getitem__(self, idx):
    img_path = self.image_paths[idx]
    image = Image.open(img_path).convert('RGB')
    if self.transform:
      image = self.transform(image)
    return image

Overwriting dataset.py


In [4]:
%%writefile test.py
import os
import numpy as np
import torch
from PIL import Image
from sklearn.metrics import precision_recall_curve, roc_auc_score
import torchvision.transforms as transforms

class tester:
  def __init__(self, modello, metal_nut_test):
    self.threshold = 0.7
    self.modello = modello
    self.metal_nut_test = metal_nut_test
    self.TP = 0
    self.FP = 0
    self.TN = 0
    self.FN = 0
    self.preprocess = transforms.Compose([                #ridimensiono croppo e normalizza l'immagine per la rete
      transforms.Resize(256),
      transforms.CenterCrop(224),
      transforms.ToTensor(),
      transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
  ])

  def testa(self):
    y_v = []
    y_p = []

    for cat in os.listdir(self.metal_nut_test):
      if cat == 'good':
        anomalia = 0
      else:
        anomalia = 1
      cat_path = os.path.join(self.metal_nut_test, cat)                #estraggo il percorso per ogni cartella
      campioni = [f for f in os.listdir(cat_path) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]      #raggruppo i campioni
      for immagine in campioni:
        img_path = os.path.join(cat_path, immagine)   #estraggo percorso per immagine
        img = Image.open(img_path)
        img_tensor = self.preprocess(img).unsqueeze(0)    #aggiungo una dime perché il linguaggio accetta solo batch
        score = self.modello.predict(img_tensor)               #calcolo il punteggio di stranezza
        y_v.append(anomalia)
        if score > self.threshold:
          y_p.append(1)
        else:
          y_p.append(0)

    for i in range(len(y_p)):                                 #calcolo le metriche
      if y_p[i] == 0 and y_v[i] == 1:
        self.FN += 1
      elif y_p[i] == 1 and y_v[i] == 0:
        self.FP += 1
      elif y_p[i] == 0 and y_v[i] == 0:
        self.TN += 1
      elif y_p[i] == 1 and y_v[i] == 1:
        self.TP += 1
    precision = self.TP / (self.TP + self.FP)
    recall = self.TP / (self.TP + self.FN)
    f1_score = 2 * (precision * recall) / (precision + recall)
    print(f"Precision: {precision}")
    print(f"Recall: {recall}")
    print(f"F1 Score: {f1_score}")


Overwriting test.py


In [5]:
import torch
import os
from dataset import immagini
from resnet import ResNet
from patchcore import PatchCore
from google.colab import drive
from torch.utils.data import DataLoader
from test import tester

drive.mount('/content/drive')

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

metal_nut = '/content/drive/MyDrive/machine/MLmigliore/data/metal_nut/train/good'
metal_nut_test = '/content/drive/MyDrive/machine/MLmigliore/data/metal_nut/test'
dataset = immagini(metal_nut)
dataloader = DataLoader(dataset, batch_size=16, shuffle=False)
estrattore = ResNet().to(device)
modello_patchcore = PatchCore(fe=estrattore, d=device)
modello_patchcore.fit(dataloader)
modello_patchcore.subsampling(0.01)
tester = tester(modello_patchcore, metal_nut_test)
tester.testa()



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
172480 vettori (patch).
1724 nuova dimensione
Precision: 0.808695652173913
Recall: 1.0
F1 Score: 0.8942307692307693


In [12]:
import os
import cv2
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

#riporta l'immagine una versione con colori visibili
def denormalize(img_tensor, mean, std):
    img = img_tensor.clone().squeeze().cpu().numpy().transpose(1, 2, 0)
    img = np.array(std) * img + np.array(mean)
    return np.clip(img, 0, 1)

dir_out = '/content/drive/MyDrive/machine/MLmigliore/data/heatmap'
os.makedirs(dir_out, exist_ok=True)
cmap = plt.get_cmap('RdYlGn_r') #carica mappa a colori
mean = [0.485, 0.456, 0.406]
std = [0.229, 0.224, 0.225]
cont = 1

for dir in os.listdir(metal_nut_test):
  path = os.path.join(metal_nut_test, dir)
  if not os.path.isdir(path) or dir.lower() == 'good':
    continue
  campioni = [f for f in os.listdir(path) if f.lower().endswith(('.png'))]

  for immagine in campioni:
    img_path = os.path.join(path, immagine)
    img = Image.open(img_path).convert('RGB')
    img_tensor = tester.preprocess(img).unsqueeze(0).to(device)
    score, heatmap = modello_patchcore.get_heatmap(img_tensor)     #ottiene heatmap dalla patchcore
    heatmap = (heatmap - np.min(heatmap)) / (np.max(heatmap) - np.min(heatmap) + 1e-8)   #normalizza

    img_rgb = denormalize(img_tensor, mean, std)
    heatmap_resized = cv2.resize(heatmap, (img_rgb.shape[1], img_rgb.shape[0]))
    heatmap_color = cmap(heatmap_resized)[:, :, :3]

    #60% nut, 40% heatmap
    overlayed_img = heatmap_color * 0.4 + img_rgb * 0.6
    overlayed_img = np.clip(overlayed_img, 0, 1)
    fig, axes = plt.subplots(1, 2, figsize=(8, 4))

    axes[0].imshow(img_rgb)
    axes[0].set_title(f"Reale: {path}", color='black', fontweight='bold')
    axes[0].axis('off')

    axes[1].imshow(overlayed_img)
    axes[1].set_title(f"Score Anomalia: {score:.2f}", color='red', fontweight='bold')
    axes[1].axis('off')

    plt.tight_layout()
    nome_file = f"{dir}_{cont}.png"
    percorso_completo = os.path.join(dir_out, nome_file)
    plt.savefig(percorso_completo, bbox_inches='tight', dpi=150)
    plt.close()

    cont += 1

print(f"end")

end


# Nuova sezione